# DyFeO3 user-executed joint CEF pipeline
Thin frontend only. Scientific algorithms live in `cef_fit`. Outputs are exploratory unless separately admitted.

## 0. Environment

In [ ]:
%pip install -e ".[cef]"
import subprocess, sys, json
from pathlib import Path


## 1. Configuration

In [ ]:
CONFIG = Path('configs/user_executed_joint_cef_pipeline_v1.yaml')
RUN_EXPLORATORY_SEARCH = False
ENABLE_JOINT_MH_FIT = False
def run_stage(stage, *extra):
    command = [sys.executable, '-m', 'cef_fit.user_pipeline', stage, '--config', str(CONFIG), *extra]
    return subprocess.run(command, check=True)


## 2. Data audit

In [ ]:
run_stage('01_data_audit')

## 3. Visualize prepared TAIPAN dataset
Visualization reads linear numerical data; rendered/log values are never passed to the fitter.

In [ ]:
import yaml, pandas as pd, matplotlib.pyplot as plt
cfg = yaml.safe_load(CONFIG.read_text())
table = pd.read_csv(cfg['inputs']['taipan_eq_dataset'])
plt.scatter(table['Q_Ainv'], table['energy_meV'], c=table['normalized_intensity'], s=8)
plt.xlabel('Q (A^-1)'); plt.ylabel('Energy (meV)'); plt.colorbar(label='linear prepared intensity');


## 4. Inspect masks

In [ ]:
from cef_fit.eq_data import load_eq_dataset, load_mask_definition, apply_masks
dataset = load_eq_dataset(cfg['inputs']['taipan_eq_dataset'], cfg['inputs']['intensity_domain'])
mask_definition = load_mask_definition(cfg['inputs']['taipan_mask_definition'])
applied_mask = apply_masks(dataset, mask_definition)
print(mask_definition); print(applied_mask.excluded_by_rule)


## 5. Forward-model sanity check

In [ ]:
run_stage('02_cef_forward')

## 6. INS objective sanity check

In [ ]:
run_stage('03_ins_objective')

## 7. Global search
The notebook switch and both configuration switches must be enabled explicitly by the user.

In [ ]:
if RUN_EXPLORATORY_SEARCH:
    assert cfg['switches']['RUN_EXPLORATORY_SEARCH'] and cfg['search']['run_exploratory_search']
    run_stage('04_global_search', '--enable-exploratory-search')
else:
    print('Global search OFF: set RUN_EXPLORATORY_SEARCH=True and enable both config switches after review.')


## 8. Search diagnostics

In [ ]:
search_summary = Path(cfg['outputs']['root']) / cfg['outputs']['search'] / 'search_summary.json'
print(search_summary.read_text() if search_summary.exists() else 'No user search output yet.')


## 9. Build INS ensemble

In [ ]:
# Run only after the visible user search completes.
# run_stage('05_build_ins_ensemble')


## 10. Wavefunctions / g tensors
Projectors are the primary invariant objects. Gauge-fixed states are presentation objects; exports occur after clustering in section 14.

In [ ]:
from cef_fit.wavefunctions import projector_overlap
print('Wavefunction kernel imported; no candidate is selected in this notebook cell.')


## 11. M(H)

In [ ]:
# run_stage('06_calculate_mh')  # frozen Blm, exact a/b 10/20 K field points


## 12. Literature M(H) comparison

In [ ]:
# run_stage('07_compare_mh')  # diagnostic only; no formal ranking


## 13. Clustering

In [ ]:
# run_stage('09_cluster_solutions')  # observable-aware preliminary families


## 14. Export representative solutions

In [ ]:
# run_stage('10_export_wavefunctions')


## 15. McPhase export

In [ ]:
# run_stage('11_export_mcphase')  # export only; independent McPhase execution remains user-visible
if ENABLE_JOINT_MH_FIT:
    raise RuntimeError('Joint M(H) fit is not authorized: accepted Fe/Dy-Fe model is absent.')
